# 04 What predicts retrieval success, and which label-free signals work?

Self-contained: this notebook downloads NFCorpus from the Hugging Face Hub, embeds it, and computes every number and figure from scratch. It does not read `results/`, `cache/`, or any `silent_rag` module, so it runs on a fresh machine with only the pip packages below. Expect roughly 15-30 minutes on a laptop CPU (embedding ~3.6k documents, then bootstrap and cross-validation loops).

**Questions**
1. Which query, document and retrieval-score features go with per-query nDCG@10?
2. Which survive multiplicity correction and cluster-aware uncertainty?
3. Can label-free features predict success on unseen topics?
4. Is the "hard topic cluster" real?

**Approach**
Spearman correlations with cluster bootstrap (design effect), Holm for the pre-specified hypotheses H1 and H2, Benjamini-Hochberg for the rest, VIF and partial correlations, topic-held-out cross-validation with a permutation null, cluster stability, and a held-out test of the lowest cluster.

**Scope.** NFCorpus only, the project's development set. SciFact and FiQA are held out for the final run, so this notebook deliberately does not load them, and the earlier cross-dataset comparisons (for example sign changes of density features between datasets) are not reproduced. H3 (window-level tracking of the top-1 score) belongs to the window-noise notebook, so the confirmatory Holm family here is H1 and H2 only, and cross-model cluster stability is not repeated here.

In [ ]:
# Uncomment on a fresh environment.
# %pip install -q datasets sentence-transformers scikit-learn scipy pandas numpy matplotlib

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # macOS: OpenMP clash between libraries; harmless elsewhere
os.environ.setdefault("OMP_NUM_THREADS", "1")

import platform
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset
from scipy import stats
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import adjusted_rand_score, r2_score, roc_auc_score, silhouette_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200, "display.max_columns", 30)

SEED = 0
np.random.seed(SEED)
K = 10
SPLITS = ["train", "validation", "test"]   # the Hub qrels name is "validation", not "dev"
GROUP_K = 50      # primary query-group definition (pre-specified); 10 and 150 are sensitivity checks
B_BOOT = 1000
MAX_SEQ = 512

NFCORPUS = dict(
    repo="BeIR/nfcorpus", revision="b5026a0e96e8a7ac4f95f482a596389289d46269",
    qrels="BeIR/nfcorpus-qrels", qrels_revision="a451b3b26d3ae1358f259c1a3a4dd61fcea35a65",
)
MODEL = dict(repo="sentence-transformers/multi-qa-MiniLM-L6-cos-v1", revision="b207367332321f8e44f96e224ef15bc607f4dbf0")

## Helpers, data and clean retrieval

Data loading, encoding, exact cosine search (top 100) and nDCG@10 (graded gain, log2 discount, as in BEIR). Every labelled query with at least one relevant document is used (train, validation and test qrels; a query keeps the first split it appears in).

In [ ]:
def doc_text(title, text):
    return f"{(title or '').strip()} {(text or '').strip()}".strip()


def load_nfcorpus():
    c = NFCORPUS
    corpus = load_dataset(c["repo"], "corpus", split="corpus", revision=c["revision"])
    queries = load_dataset(c["repo"], "queries", split="queries", revision=c["revision"])
    qtext = {str(q["_id"]): q["text"] for q in queries}
    split_of, qrels = {}, {}
    for split in SPLITS:
        for row in load_dataset(c["qrels"], split=split, revision=c["qrels_revision"]):
            q = str(row["query-id"])
            if split_of.setdefault(q, split) == split:
                qrels.setdefault(q, {})[str(row["corpus-id"])] = int(row["score"])
    doc_ids = [str(d["_id"]) for d in corpus]
    docs = [doc_text(t, x) for t, x in zip(corpus["title"], corpus["text"])]
    return doc_ids, docs, {q: qtext[q] for q in qrels}, qrels, split_of


def ndcg_at_k(ranked, rels, k=K):
    dcg = sum(rels.get(d, 0) / np.log2(i + 2) for i, d in enumerate(ranked[:k]))
    ideal = sorted(rels.values(), reverse=True)[:k]
    idcg = sum(r / np.log2(i + 2) for i, r in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0


def encode(enc, texts, batch_size=32):
    return np.ascontiguousarray(enc.encode(list(texts), batch_size=batch_size, normalize_embeddings=True,
                                           convert_to_numpy=True, show_progress_bar=False), dtype=np.float32)


def search(D, Q, depth=100):
    S = Q @ D.T
    # Stable full sort: exact score ties (the corpus contains duplicate documents) break by document index,
    # so the top-k does not depend on which search routine is used.
    top = np.argsort(-S, axis=1, kind="stable")[:, :depth]
    return top, np.take_along_axis(S, top, axis=1)


doc_ids, docs, queries, qrels, split_of = load_nfcorpus()
enc = SentenceTransformer(MODEL["repo"], revision=MODEL["revision"], device="cpu")
enc.max_seq_length = MAX_SEQ

qids = [q for q in queries if any(r > 0 for r in qrels[q].values())]
split = np.array([split_of[q] for q in qids])
rel = [{d for d, v in qrels[q].items() if v > 0} for q in qids]
D = encode(enc, docs)
Q = encode(enc, [queries[q] for q in qids])
top, scores = search(D, Q, depth=100)
ids = [[doc_ids[j] for j in row] for row in top]
ndcg = np.array([ndcg_at_k(r, qrels[q]) for r, q in zip(ids, qids)])
recall = np.array([len(r & set(i[:K])) / len(r) for r, i in zip(rel, ids)])
print(len(qids), "queries | mean nDCG@10", round(float(ndcg.mean()), 4), "| share of zeros", round(float((ndcg == 0).mean()), 3))

## Features

| Feature | Label-free | Notes |
|---|---|---|
| `q_tokens` | yes | query length |
| `top1_score`, `top10_mean_score`, `top1_top2_gap` | yes | retrieval scores |
| `nn_query_sim`, `q_density` | yes | nearest and mean-of-10 nearest other query, over all labelled queries (transductive; recomputed inside training folds for the CV section) |
| `n_rel` | no | relevant documents |
| `rel_doc_tokens`, `rel_frac_over_256` | no | length of relevant documents |
| `rel_cos` | no | cosine of query to its relevant documents (near-outcome by construction) |
| `rel_hubness_loo` | no | mean k-occurrence of the relevant documents, **excluding the query's own top-10 contribution** |

`rel_hubness_orig` keeps the earlier definition only to quantify the circularity.

"Label-free" means the feature uses no qrels. Note, though, that the query set itself is "labelled queries with at least one relevant document", a selection a production system would not make.

In [ ]:
FEATURES = {"q_tokens": True, "top1_score": True, "top10_mean_score": True, "top1_top2_gap": True, "nn_query_sim": True, "q_density": True,
            "n_rel": False, "rel_doc_tokens": False, "rel_frac_over_256": False, "rel_cos": False, "rel_hubness_loo": False}
LABEL_FREE = [f for f, v in FEATURES.items() if v]

pos = {d: i for i, d in enumerate(doc_ids)}
tok = enc.tokenizer
kocc = np.bincount(top[:, :K].ravel(), minlength=len(D))
S = Q @ Q.T
np.fill_diagonal(S, -np.inf)
nn_top = -np.sort(-S, axis=1)[:, :K]
rel_docs = sorted({d for r in rel for d in r if d in pos})
doc_len = {d: len(tok.tokenize(docs[pos[d]])) for d in rel_docs}

rows = []
for i, r in enumerate(rel):
    r = [d for d in r if d in pos]
    own = set(top[i, :K])
    lens = np.array([doc_len[d] for d in r])
    rows.append({
        "q_tokens": len(tok.tokenize(queries[qids[i]])), "top1_score": scores[i, 0], "top10_mean_score": scores[i, :K].mean(),
        "top1_top2_gap": scores[i, 0] - scores[i, 1], "nn_query_sim": nn_top[i, 0], "q_density": nn_top[i].mean(),
        "n_rel": len(r), "rel_doc_tokens": lens.mean(), "rel_frac_over_256": (lens > 256).mean(),
        "rel_cos": float((D[[pos[d] for d in r]] @ Q[i]).mean()),
        "rel_hubness_orig": float(np.mean([kocc[pos[d]] for d in r])),
        "rel_hubness_loo": float(np.mean([kocc[pos[d]] - (pos[d] in own) for d in r])),
    })
df = pd.DataFrame(rows)
df["ndcg10"] = ndcg
df["failed"] = (recall == 0).astype(int)
df.describe().T.round(3)

## Query groups and the design effect

Queries on the same topic, or sharing relevant documents, are not independent, so uncertainty is estimated by resampling **groups** of queries. Three group definitions: KMeans on the query embeddings with k = 10, 50 (primary) and 150, plus connected components of queries that share a relevant document. Group sizes show how strong the dependence is. A single giant component would mean shared-document structure links most queries, and that definition would be unusable.

In [ ]:
def components_by_shared_docs(rel_sets):
    parent = list(range(len(rel_sets)))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    owner = {}
    for i, r in enumerate(rel_sets):
        for d in r:
            if d in owner:
                parent[find(i)] = find(owner[d])
            else:
                owner[d] = i
    return np.array([find(i) for i in range(len(rel_sets))])


groups = {f"kmeans{k}": KMeans(k, n_init=5, random_state=SEED).fit_predict(Q) for k in (10, GROUP_K, 150)}
groups["shared_doc_components"] = components_by_shared_docs(rel)
rows = []
for gname, lab in groups.items():
    sizes = pd.Series(lab).value_counts()
    rows.append({"grouping": gname, "n_groups": len(sizes), "largest_group_share": sizes.iloc[0] / len(lab), "median_size": sizes.median()})
pd.DataFrame(rows).round(3)

## Univariate associations with cluster bootstrap

Spearman rho between each feature and nDCG@10. The CI resamples whole groups (primary: KMeans k = 50) `B = 1000` times. The p-value is the two-sided bootstrap p, `(1 + 2 * min(#{rho* <= 0}, #{rho* >= 0})) / (B + 1)`, and the plain Spearman p (independence assumed) is shown for contrast. The **design effect** is the cluster-CI width divided by the iid-CI width: values above 1 mean the independent-query CI was too narrow. A bootstrap replicate whose Spearman is undefined (a constant resample) is dropped; the number dropped is counted and reported, never hidden.

**Why this test**
- **Estimand:** the population Spearman correlation between a feature and per-query nDCG@10 over NFCorpus queries under this model.
- **Unit of independence:** the KMeans(50) topic group, not the query. Queries on one topic share relevant documents (notebook 01), so query-level resampling understates the variance. Groups are a proxy for the unknown source page, so these CIs are lower bounds (see the sensitivity table below).
- **Assumptions:** groups are roughly exchangeable and independent of one another. With 50 groups the percentile bootstrap is adequate. The grouping-sensitivity and KMeans-seed tables check it.
- **Why not the obvious alternative:** the analytic Spearman p (`p_plain`) assumes independent queries, and nDCG is bounded, zero-inflated (33% zeros) and tied, so a Fisher-z interval is not reliable. A cluster bootstrap needs no distributional form.
- **Resolution / Monte Carlo error:** the bootstrap p-value floor is `1/(B+1) = 0.001` for B = 1000. A p at the floor is reported as `p < 0.001`, not as an exact value.

In [ ]:
def spearman(x, y):
    return stats.spearmanr(x, y)[0]


NAN_DROPPED = {"total": 0}   # undefined bootstrap replicates are counted, not silently discarded


def boot_rho(x, y, labels, B=B_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    if labels is None:
        n = len(x)
        out = [spearman(x[i], y[i]) for i in (rng.integers(0, n, n) for _ in range(B))]
    else:
        uniq, inv = np.unique(labels, return_inverse=True)
        members = [np.where(inv == g)[0] for g in range(len(uniq))]
        out = []
        for _ in range(B):
            idx = np.concatenate([members[g] for g in rng.integers(0, len(uniq), len(uniq))])
            out.append(spearman(x[idx], y[idx]))
    out = np.array(out)
    boot_rho.last_dropped = int(np.isnan(out).sum())
    NAN_DROPPED["total"] += boot_rho.last_dropped
    return out[~np.isnan(out)]


def boot_p(boot):
    """Two-sided bootstrap p for rho = 0: (1 + 2 * smaller tail count) / (B + 1)."""
    return min(1.0, (1 + 2 * min((boot <= 0).sum(), (boot >= 0).sum())) / (len(boot) + 1))


def boot_p_pos(boot):
    """One-sided bootstrap p for rho > 0: (1 + #{rho* <= 0}) / (B + 1)."""
    return (1 + (boot <= 0).sum()) / (len(boot) + 1)


y = df["ndcg10"].to_numpy()
lab = groups[f"kmeans{GROUP_K}"]
rows = []
for feat in list(FEATURES) + ["rel_hubness_orig"]:
    x = df[feat].to_numpy(float)
    if np.ptp(x) == 0:
        continue
    rho, p_plain = stats.spearmanr(x, y)
    b_iid = boot_rho(x, y, None)
    drop_iid = boot_rho.last_dropped
    b_cl = boot_rho(x, y, lab)
    drop_cl = boot_rho.last_dropped
    ci_iid, ci_cl = np.percentile(b_iid, [2.5, 97.5]), np.percentile(b_cl, [2.5, 97.5])
    rows.append({"feature": feat, "label_free": FEATURES.get(feat, False), "n": len(y), "rho": rho,
                 "ci_lo_iid": ci_iid[0], "ci_hi_iid": ci_iid[1], "ci_lo_cluster": ci_cl[0], "ci_hi_cluster": ci_cl[1],
                 "design_effect": (ci_cl[1] - ci_cl[0]) / (ci_iid[1] - ci_iid[0]), "p_plain": p_plain, "p_cluster": boot_p(b_cl),
                 "p_cluster_onesided_pos": boot_p_pos(b_cl), "nan_dropped_iid": drop_iid, "nan_dropped_cluster": drop_cl})
uni = pd.DataFrame(rows)
print("design effect (cluster CI width / iid CI width): median", round(float(uni["design_effect"].median()), 2), "max", round(float(uni["design_effect"].max()), 2))
print("undefined (NaN) bootstrap replicates dropped in this table:", int(uni[["nan_dropped_iid", "nan_dropped_cluster"]].to_numpy().sum()),
      "of", 2 * B_BOOT * len(uni))
uni.round(4)

## Multiplicity

**Confirmatory (Holm):** H1 (rho of the top-1 score with nDCG@10 is positive) and H2 (rho of the top-1 minus top-2 gap is positive), using the one-sided cluster-bootstrap p `(1 + #{rho* <= 0}) / (B + 1)`. The bootstrap p has a floor of 1/(B+1) ≈ 0.001, so Holm-adjusted values cannot go below about 0.002. Values at the floor mean `p < 0.002`, not an exact p. **Exploratory (Benjamini-Hochberg, FDR 5%):** every two-sided cluster-bootstrap p, excluding `rel_hubness_orig`.

**Why this test**
- **Estimand:** for H1 and H2, the sign of rho for the two signals named as hypotheses in the project's analysis plan (that plan is not under version control, so it cannot be shown to predate these analyses, and held-out results had already been seen; see the README); for the exploratory rows, which other associations are distinguishable from zero.
- **Unit of independence:** the KMeans(50) group (inherited from the bootstrap above).
- **Assumptions:** Holm controls the familywise error rate under any dependence between the two tests (top-1 and gap are correlated). BH controls FDR under positive dependence, which is plausible for correlated score features but not guaranteed.
- **Why not the obvious alternative:** Bonferroni is uniformly less powerful than Holm with the same guarantee. Putting all 11 features in the confirmatory family would dilute the two hypotheses. Note that H1 is near-mechanical (top-1 and nDCG come from the same retrieval), so its *size* (rho) is the content, not the rejection.
- **Resolution / Monte Carlo error:** floor `1/(B+1)` per test; at the floor, the Holm value for the family of two is about 0.002.

In [ ]:
def holm(p):
    p = np.asarray(p, float)
    order = np.argsort(p)
    out, run = np.empty(len(p)), 0.0
    for rank, i in enumerate(order):
        run = max(run, min(1.0, p[i] * (len(p) - rank)))
        out[i] = run
    return out


def bh(p):
    p = np.asarray(p, float)
    order = np.argsort(p)
    ranked = p[order] * len(p) / (np.arange(len(p)) + 1)
    adj = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty(len(p))
    out[order] = np.minimum(adj, 1.0)
    return out


conf = uni[uni["feature"].isin(["top1_score", "top1_top2_gap"])][["feature", "rho", "ci_lo_cluster", "ci_hi_cluster", "p_cluster_onesided_pos"]].copy()
conf["hypothesis"] = conf["feature"].map({"top1_score": "H1", "top1_top2_gap": "H2"})
conf["p_holm"] = holm(conf["p_cluster_onesided_pos"])
conf["reject_at_0.05"] = conf["p_holm"] < 0.05
display(conf.round(4))

exp = uni[uni["feature"] != "rel_hubness_orig"].copy()
exp["p_bh"] = bh(exp["p_cluster"])
exp["significant_fdr05"] = exp["p_bh"] < 0.05
exp[["feature", "label_free", "rho", "ci_lo_cluster", "ci_hi_cluster", "design_effect", "p_bh", "significant_fdr05"]].round(4)

In [ ]:
order_ = exp.sort_values("rho")
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.barh(order_["feature"], order_["rho"], color=np.where(order_["label_free"], "tab:blue", "lightgrey"),
        xerr=[order_["rho"] - order_["ci_lo_cluster"], order_["ci_hi_cluster"] - order_["rho"]], capsize=2)
ax.axvline(0, color="grey")
ax.set_xlabel("Spearman rho with nDCG@10 (95% cluster CI); blue = label-free")
fig.tight_layout()
plt.show()

## Sensitivity of the CIs to the group definition

The two confirmatory features under every grouping (and the plain bootstrap). If the conclusion changes with the grouping, the grouping choice matters and should be stated.

**Why this test**
- **Estimand:** how much the interval for rho depends on the assumed unit of independence.
- **Unit of independence:** varied deliberately: query (iid), KMeans k = 150, 50 and 10, and shared-document components.
- **Assumptions:** none beyond the bootstrap's own. A widening interval as groups coarsen is evidence that dependence extends beyond the finer grouping.
- **Why not the obvious alternative:** a single "best" grouping cannot be justified without source-page metadata, so the range is the honest summary. The widest *usable* grouping (k = 10) is the conservative interval to quote.
- **Resolution / Monte Carlo error:** 500 replicates per row; the Monte Carlo SE of a 2.5% percentile endpoint is roughly 0.005 in rho.

In [ ]:
rows = []
for feat in ("top1_score", "top1_top2_gap"):
    x = df[feat].to_numpy(float)
    rho = spearman(x, y)
    for gname, lab_ in [("iid", None)] + list(groups.items()):
        b = boot_rho(x, y, lab_, B=500)
        lo, hi = np.percentile(b, [2.5, 97.5])
        rows.append({"feature": feat, "grouping": gname, "rho": rho, "ci_lo": lo, "ci_hi": hi, "width": hi - lo, "nan_dropped": boot_rho.last_dropped})
pd.DataFrame(rows).round(3)

## Stability of the topic grouping (KMeans seed)

The primary groups use KMeans seed 0. This repeats the confirmatory CIs with five different seeds. The spread is the part of the uncertainty that comes from the arbitrary grouping.

In [ ]:
rows = []
for gs in range(5):
    lab_s = KMeans(GROUP_K, n_init=5, random_state=gs).fit_predict(Q)
    for feat in ("top1_score", "top1_top2_gap"):
        x = df[feat].to_numpy(float)
        b = boot_rho(x, y, lab_s, B=500)
        lo, hi = np.percentile(b, [2.5, 97.5])
        rows.append({"grouping_seed": gs, "feature": feat, "rho": spearman(x, y), "ci_lo": lo, "ci_hi": hi,
                     "design_effect": (hi - lo) / np.ptp(np.percentile(boot_rho(x, y, None, B=500), [2.5, 97.5]))})
t = pd.DataFrame(rows)
t.groupby("feature").agg(rho=("rho", "first"), ci_lo_min=("ci_lo", "min"), ci_lo_max=("ci_lo", "max"), ci_hi_min=("ci_hi", "min"),
                         ci_hi_max=("ci_hi", "max"), design_effect_min=("design_effect", "min"), design_effect_max=("design_effect", "max")).round(3)

## The circularity of `rel_hubness`

The original definition counts a relevant document's retrievals including the query's own. If a relevant document is retrieved for its own query it contributes to its own k-occurrence, so the feature partly encodes the outcome. Comparison of the original and leave-one-out versions (cluster CIs, from the table above):

In [ ]:
uni[uni["feature"].isin(["rel_hubness_orig", "rel_hubness_loo"])][["feature", "rho", "ci_lo_cluster", "ci_hi_cluster"]].round(3)

## Collinearity and partial correlations

Variance inflation factors (VIF = 1 / (1 - R^2) of each feature regressed on the others, on ranks) quantify the collinearity that makes regression coefficients hard to interpret. Partial Spearman correlations (ranks residualized on the control set) separate the score signals from each other: `top1_score` given `top1_top2_gap` and `top10_mean_score`, and `top1_top2_gap` given the other two. Cluster-bootstrap CIs.

**Why this test**
- **Estimand:** the rank association between a signal and nDCG@10 that remains after linear adjustment (on ranks) for the other signals.
- **Unit of independence:** the KMeans(50) group (cluster bootstrap, 300 replicates).
- **Assumptions:** the confounding is approximately linear in ranks. VIF is descriptive and needs no distributional assumption.
- **Why not the obvious alternative:** regression coefficients on collinear features (VIF 9–12) are unstable and sign-flip easily. A partial correlation answers the narrower, stable question "does this signal add anything given the others?".
- **Resolution / Monte Carlo error:** 300 replicates; the CI endpoints carry a Monte Carlo SE of about 0.01.

In [ ]:
X = df[list(FEATURES)].rank().to_numpy()
X = (X - X.mean(0)) / (X.std(0) + 1e-12)
vif = []
for j, f in enumerate(FEATURES):
    others = np.delete(X, j, axis=1)
    r2 = LinearRegression().fit(others, X[:, j]).score(others, X[:, j])
    vif.append({"feature": f, "VIF": 1 / max(1 - r2, 1e-9)})
display(pd.DataFrame(vif).round(1))


def partial_spearman(frame, x, controls, y="ndcg10"):
    R = frame[[x, y] + controls].rank()
    C = np.column_stack([np.ones(len(R)), R[controls].to_numpy()])
    res = lambda v: v - C @ np.linalg.lstsq(C, v, rcond=None)[0]
    return np.corrcoef(res(R[x].to_numpy()), res(R[y].to_numpy()))[0, 1]


uniq, inv = np.unique(groups[f"kmeans{GROUP_K}"], return_inverse=True)
members = [np.where(inv == g)[0] for g in range(len(uniq))]
rng = np.random.default_rng(SEED)
specs = [("top1_score", ["top1_top2_gap", "top10_mean_score"]), ("top1_top2_gap", ["top1_score", "top10_mean_score"]),
         ("top1_score", ["n_rel", "q_tokens"]), ("q_density", ["top1_score", "top1_top2_gap"])]
rows = []
for x_, ctrl in specs:
    est = partial_spearman(df, x_, ctrl)
    boots = [partial_spearman(df.iloc[np.concatenate([members[g] for g in rng.integers(0, len(uniq), len(uniq))])], x_, ctrl) for _ in range(300)]
    rows.append({"feature": x_, "controlling_for": "+".join(ctrl), "partial_rho": est,
                 "ci_lo": np.percentile(boots, 2.5), "ci_hi": np.percentile(boots, 97.5)})
pd.DataFrame(rows).round(3)

## Out-of-sample prediction with topic-held-out folds

Can the features **predict** nDCG@10 for queries on topics the model has not seen? 5-fold `GroupKFold` with groups from KMeans k = 50, repeated over 5 different group assignments. Models: a ridge regression and a gradient-boosted tree model (non-linear, interactions), each on the **label-free** features and on **all** features.

**Two corrections in this revision.**
1. *Point estimate and CI now describe the same quantity.* The out-of-fold predictions of the 5 repeats are averaged per query (`pm`), and the table reports `r2_of_mean_pred = R^2(y, pm)` together with its group-bootstrap CI. The mean of the 5 per-repeat R^2 values is kept as a separate, labelled column (`r2_mean_of_repeats`); it is a different statistic and has no CI here.
2. *Fold-internal density features.* `nn_query_sim` and `q_density` were computed over all queries, so a held-out query's feature used the training queries *and* the other held-out queries (transductive). In the `fold-internal` rows they are recomputed per fold: a training query's neighbours come from the other training-fold queries, and a held-out query's neighbours come from training-fold queries only. The `transductive` rows keep the old definition for comparison.

A coarser row holds out KMeans k = 10 groups (fold-internal features), because neighbouring k = 50 clusters can share topics across folds. Reported: R^2 of the averaged out-of-fold prediction with a 95% group-bootstrap CI (groups of the same k as the folds), and the AUROC of the same averaged prediction for failure (recall@10 = 0). A permutation test (100 shuffles of nDCG across queries, ridge, label-free, fold-internal) gives the p-value for "R^2 > 0".

**Assumption checks.** (1) Linearity: if the gradient-boosted model beats ridge out of sample, linearity was too strong. (2) nDCG is bounded and zero-inflated, so residuals are plotted for the ridge fit.

**Why this test**
- **Estimand:** the predictive R^2 of label-free features for nDCG@10 on queries from topic groups not seen in training.
- **Unit of independence:** the topic group: folds and the CI both resample whole groups.
- **Assumptions:** held-out groups are exchangeable with training groups. Features use no information from the held-out fold (now true for the density features as well).
- **Why not the obvious alternative:** random K-fold CV would put near-duplicate and same-topic queries on both sides of a split and inflate R^2. In-sample R^2 measures fit, not prediction.
- **Resolution / Monte Carlo error:** 300 bootstrap replicates for the CI. The permutation p floor is `1/101 ≈ 0.0099`, so a p at the floor means `p < 0.01`.

In [ ]:
def make_model(kind):
    if kind == "ridge":
        return Ridge(alpha=1.0)
    return HistGradientBoostingRegressor(max_depth=3, max_iter=150, learning_rate=0.05, random_state=SEED)


DENS_COLS = ("nn_query_sim", "q_density")
_dens_cache = {}


def fold_density(tr, te):
    """nn_query_sim and q_density using training-fold queries only (a training query excludes itself)."""
    key = te.tobytes()
    if key not in _dens_cache:
        S_tr = Q[tr] @ Q[tr].T
        np.fill_diagonal(S_tr, -np.inf)
        top_tr = -np.sort(-S_tr, axis=1)[:, :K]
        top_te = -np.sort(-(Q[te] @ Q[tr].T), axis=1)[:, :K]
        _dens_cache[key] = ({"nn_query_sim": top_tr[:, 0], "q_density": top_tr.mean(1)},
                            {"nn_query_sim": top_te[:, 0], "q_density": top_te.mean(1)})
    return _dens_cache[key]


def fold_X(X, feats, tr, te, fold_internal):
    X_tr, X_te = X[tr].copy(), X[te].copy()
    if fold_internal:
        d_tr, d_te = fold_density(tr, te)
        for c in DENS_COLS:
            if c in feats:
                j = feats.index(c)
                X_tr[:, j], X_te[:, j] = d_tr[c], d_te[c]
    return X_tr, X_te


def oof_predict(X, y, groups_, kind, feats=None, fold_internal=False):
    pred = np.empty(len(y))
    for tr, te in GroupKFold(5).split(X, y, groups_):
        X_tr, X_te = fold_X(X, feats or [], tr, te, fold_internal)
        sc = StandardScaler().fit(X_tr)
        pred[te] = make_model(kind).fit(sc.transform(X_tr), y[tr]).predict(sc.transform(X_te))
    return pred


def group_boot(metric, y_, pred, labels, B=300, seed=SEED):
    uniq_, inv_ = np.unique(labels, return_inverse=True)
    mem = [np.where(inv_ == g)[0] for g in range(len(uniq_))]
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(B):
        idx = np.concatenate([mem[g] for g in rng.integers(0, len(uniq_), len(uniq_))])
        try:
            out.append(metric(y_[idx], pred[idx]))
        except ValueError:
            pass
    return np.percentile(out, [2.5, 97.5]), B - len(out)


failed = df["failed"].to_numpy()
_km = {}


def kmeans_rep(k, rep):
    if (k, rep) not in _km:
        _km[(k, rep)] = KMeans(k, n_init=3, random_state=SEED + rep).fit_predict(Q)
    return _km[(k, rep)]


specs = [(fs, kind, var, GROUP_K) for fs in ("label-free", "all") for kind in ("ridge", "gbm") for var in ("transductive", "fold-internal")]
specs += [("label-free", kind, "fold-internal", 10) for kind in ("ridge", "gbm")]
cv_rows, oof_store = [], {}
for fs_name, kind, var, k in specs:
    feats = LABEL_FREE if fs_name == "label-free" else list(FEATURES)
    Xf = df[feats].to_numpy(float)
    r2s, preds = [], []
    for rep in range(5):
        p = oof_predict(Xf, y, kmeans_rep(k, rep), kind, feats, fold_internal=(var == "fold-internal"))
        r2s.append(r2_score(y, p)); preds.append(p)
    pm = np.mean(preds, axis=0)
    (lo, hi), n_fail_boot = group_boot(r2_score, y, pm, kmeans_rep(k, 0))
    oof_store[(fs_name, kind, var, k)] = pm
    cv_rows.append({"features": fs_name, "model": kind, "density_features": var, "fold_groups": f"kmeans{k}",
                    "r2_of_mean_pred": r2_score(y, pm), "r2_ci_lo": lo, "r2_ci_hi": hi,
                    "r2_mean_of_repeats": np.mean(r2s), "r2_sd_over_repeats": np.std(r2s, ddof=1),
                    "auroc_failure_of_mean_pred": roc_auc_score(failed, -pm), "boot_replicates_failed": n_fail_boot})
cv = pd.DataFrame(cv_rows)
cv.round(4)

In [ ]:
# permutation test: is out-of-fold R^2 (ridge, label-free, fold-internal density features) above zero?
# The features do not depend on y, so the per-fold density cache is reused across permutations.
lab = groups[f"kmeans{GROUP_K}"]
Xlf = df[LABEL_FREE].to_numpy(float)
obs = r2_score(y, oof_predict(Xlf, y, lab, "ridge", LABEL_FREE, fold_internal=True))
rng = np.random.default_rng(SEED)
null = np.array([r2_score(y, oof_predict(Xlf, rng.permutation(y), lab, "ridge", LABEL_FREE, fold_internal=True)) for _ in range(100)])
pd.DataFrame([{"oof_r2": obs, "null_mean": null.mean(), "null_p95": np.percentile(null, 95), "perm_p": (1 + (null >= obs).sum()) / 101}]).round(4)

In [ ]:
# permutation importance of the label-free features, measured on held-out folds (gbm, fold-internal density features).
# Under collinearity (VIF 9-12 for the score features) permuting one feature leaves its correlated sibling, so the split of
# credit between top1_score and top10_mean_score is not interpretable; a grouped drop of all three score features is shown too.
imp = np.zeros((5, len(LABEL_FREE)))
grp_drop = np.zeros(5)
score_cols = [LABEL_FREE.index(c) for c in ("top1_score", "top10_mean_score", "top1_top2_gap")]
for f, (tr, te) in enumerate(GroupKFold(5).split(Xlf, y, lab)):
    X_tr, X_te = fold_X(Xlf, LABEL_FREE, tr, te, True)
    sc = StandardScaler().fit(X_tr)
    m = make_model("gbm").fit(sc.transform(X_tr), y[tr])
    imp[f] = permutation_importance(m, sc.transform(X_te), y[te], n_repeats=10, random_state=SEED, scoring="r2").importances_mean
    keep = [j for j in range(len(LABEL_FREE)) if j not in score_cols]
    sc2 = StandardScaler().fit(X_tr[:, keep])
    m2 = make_model("gbm").fit(sc2.transform(X_tr[:, keep]), y[tr])
    grp_drop[f] = r2_score(y[te], m.predict(sc.transform(X_te))) - r2_score(y[te], m2.predict(sc2.transform(X_te[:, keep])))
display(pd.DataFrame({"feature": LABEL_FREE, "r2_drop_mean": imp.mean(0), "r2_drop_sd_over_folds": imp.std(0, ddof=1)}).round(4))
print("drop-column importance of the three score features together (held-out R^2 lost):",
      round(float(grp_drop.mean()), 4), "+/-", round(float(grp_drop.std(ddof=1)), 4), "(SD over folds)")

In [ ]:
p = oof_store[("label-free", "ridge", "fold-internal", GROUP_K)]
fig, ax = plt.subplots(figsize=(4, 3))
ax.scatter(p, y - p, s=3, alpha=0.3)
ax.axhline(0, color="grey")
ax.set_title("ridge residuals (out-of-fold, label-free, fold-internal density)", fontsize=8)
ax.set_xlabel("predicted nDCG@10")
ax.set_ylabel("residual")
fig.tight_layout()
plt.show()
print("NaN bootstrap replicates dropped by boot_rho across the whole notebook so far:", NAN_DROPPED["total"])

## Cluster stability across seeds and k

20 KMeans seeds per k on the query embeddings. Stability is the mean **adjusted Rand index (ARI)** of each seed's partition against the seed-0 partition (1 = identical, 0 = chance agreement). **Silhouette** (cosine, 2,000-query subsample) measures how well separated the clusters are. Low silhouette with high ARI means the partition is reproducible but the clusters are not sharply separated, which is the usual case for text embeddings and means cluster identity is a soft description.

In [ ]:
sub = np.random.default_rng(SEED).choice(len(Q), min(2000, len(Q)), replace=False)
rows = []
for k in (5, 10, 20, 50):
    labs = [KMeans(k, n_init=3, random_state=s).fit_predict(Q) for s in range(20)]
    ari = [adjusted_rand_score(labs[0], l) for l in labs[1:]]
    sil = [silhouette_score(Q[sub], l[sub], metric="cosine") for l in labs[:5]]
    rows.append({"k": k, "ARI_vs_seed0_mean": np.mean(ari), "ARI_min": np.min(ari), "silhouette_mean": np.mean(sil)})
pd.DataFrame(rows).round(3)

## Is the lowest cluster really low?

Picking the worst of k clusters and testing it as if it had been chosen in advance overstates significance. Two corrections:

- **Permutation of the minimum.** Statistic: the most negative standardized cluster deviation `min_c (mean_c - mean) / (sd / sqrt(n_c))`. Null: shuffle nDCG across queries (2,000 times) and take the same minimum. The p-value is the share of null minima at least as extreme.
- **Held-out check.** On each of 20 seeds, choose the lowest cluster using **train queries only**, then compare that cluster's nDCG with the rest on the **held-out queries** (validation and test), with a bootstrap CI of the difference. The fraction of seeds with a negative held-out difference, and its median, say whether the "hard cluster" replicates on queries that played no part in choosing it.

**Why this test**
- **Estimand:** whether the lowest-scoring topic cluster is lower than selection alone would produce, and whether that holds on queries not used to pick it.
- **Unit of independence:** query, for the permutation of the minimum (it breaks the cluster–nDCG link); query within split, for the held-out check.
- **Assumptions:** under the null, nDCG is exchangeable across clusters. The held-out check assumes held-out queries are independent of training queries, which is **false** here: they share clusters and relevant documents (notebook 01), so replication is partly a shared-document effect.
- **Why not the obvious alternative:** a t-test on the chosen cluster ignores that it was selected as the minimum of k (the winner's curse). Kruskal-Wallis is shown only as an omnibus check, and it treats queries as independent.
- **Resolution / Monte Carlo error:** permutation floor `1/2001 ≈ 0.0005`; held-out CIs from 1,000 query-bootstrap replicates (iid, so too narrow).

In [ ]:
def min_z(nd, labels):
    z = []
    for c in np.unique(labels):
        m = labels == c
        if m.sum() > 5:
            z.append((nd[m].mean() - nd.mean()) / (nd.std(ddof=1) / np.sqrt(m.sum())))
    return min(z)


rows = []
for k in (10, 50):
    lab_k = KMeans(k, n_init=5, random_state=SEED).fit_predict(Q)
    obs_z = min_z(ndcg, lab_k)
    rng = np.random.default_rng(SEED)
    null = np.array([min_z(rng.permutation(ndcg), lab_k) for _ in range(2000)])
    kw = stats.kruskal(*[ndcg[lab_k == c] for c in np.unique(lab_k)])
    rows.append({"k": k, "min_z": obs_z, "perm_p_min": (1 + (null <= obs_z).sum()) / 2001, "kruskal_H": kw.statistic, "kruskal_p": kw.pvalue})
display(pd.DataFrame(rows).round(4))

train, hold = split == "train", split != "train"
diffs = []
for seed in range(20):
    lab_s = KMeans(10, n_init=3, random_state=seed).fit_predict(Q)
    means = {c: ndcg[train & (lab_s == c)].mean() for c in np.unique(lab_s) if (train & (lab_s == c)).sum() >= 10}
    low = min(means, key=means.get)
    inside, outside = ndcg[hold & (lab_s == low)], ndcg[hold & (lab_s != low)]
    if len(inside) < 10:
        continue
    rng = np.random.default_rng(seed)
    b = [inside[rng.integers(0, len(inside), len(inside))].mean() - outside[rng.integers(0, len(outside), len(outside))].mean() for _ in range(1000)]
    diffs.append({"seed": seed, "low_cluster_train_mean": means[low], "heldout_inside": inside.mean(), "heldout_outside": outside.mean(),
                  "heldout_diff": inside.mean() - outside.mean(), "ci_lo": np.percentile(b, 2.5), "ci_hi": np.percentile(b, 97.5), "n_inside": len(inside)})
d = pd.DataFrame(diffs)
pd.DataFrame([{"seeds_used": len(d), "median_heldout_diff": d["heldout_diff"].median(), "share_seeds_negative": (d["heldout_diff"] < 0).mean(),
               "share_seeds_ci_excludes_0_negative": (d["ci_hi"] < 0).mean(), "median_train_mean_of_low_cluster": d["low_cluster_train_mean"].median(),
               "median_n_heldout_inside": d["n_inside"].median()}]).round(4)

In [ ]:
import datasets, sentence_transformers, sklearn, scipy, torch
print({"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__,
       "sklearn": sklearn.__version__, "torch": torch.__version__, "sentence_transformers": sentence_transformers.__version__,
       "datasets": datasets.__version__, "seed": SEED, "device": "cpu", "max_seq_length": MAX_SEQ, "group_k": GROUP_K, "bootstrap_B": B_BOOT})

## Reproducibility check against the original analysis

The original analysis saved its results as tables under `results/tables/`. This notebook recomputes everything from scratch and was compared with those tables. Where a value differs, the cause was traced and, where it was a flaw in the method rather than a rounding effect, fixed here. **Two sources of tiny differences apply to every notebook:** (1) the corpus contains 39 pairs of identical document vectors, so 29 queries have an exact score tie at rank 10/11, and which document is kept depends on the search routine (mean nDCG@10 ranges 0.29581 to 0.29590 across routines). This notebook breaks ties by document index (stable sort), which makes the top-k deterministic; the project's `pipeline.Retriever` uses FAISS, so its clean baseline can differ from this notebook in the 4th decimal. (2) The package versions differ from those that wrote the saved tables, so differences in the third decimal are expected.

| Quantity | Original | This notebook | Status |
|---|---|---|---|
| H1: rho(top-1 score, nDCG) and cluster CI | 0.522 (to 0.571), design effect 2.01 | 0.522 (0.467 to 0.571), design effect 2.02 | identical |
| H2: rho(top-1 gap, nDCG) | 0.199 | 0.199 (0.157 to 0.239) | identical |
| Partial rho of top-1, given gap and top-10 mean | 0.330 | 0.330 | identical |
| VIF of top-1 / top-10 mean | 8.6 / 11.6 | 8.6 / 11.6 | identical |
| Out-of-fold R^2, label-free ridge / gbm (mean of 5 repeats, transductive density) | 0.2753 / 0.3106 | 0.2754 / 0.3100 | +/- 0.0006 |
| Permutation p for out-of-fold R^2 | 0.0099 | 0.0099 (now with fold-internal density features) | identical (resolution floor) |
| Kruskal H, k = 10 / 50 | 148.60 / 280.20 | 148.29 / 279.97 | -0.2% / -0.1% |
| Held-out difference of the lowest cluster | -0.1319 | -0.1324 | -0.4% |
| **Holm-adjusted p for H1 and H2** | 0.003 / 0.003 | 0.002 / 0.002 | **changed** |
| **R^2 reported with its CI** | 0.275 with CI 0.234 to 0.313 (different quantities) | ridge 0.284 (0.238 to 0.322), gbm 0.311 (0.266 to 0.354), same quantity | **changed (second review)** |
| **Bootstrap p-value formula** | `2 · tail share + 1/B` | `(1 + 2 · tail count) / (B + 1)`; for example query tokens two-sided p = 0.0749 | changed formula, same conclusions |

**Diagnosis.**
1. *Holm family size.* The original corrected H1, H2 and H3 together; H3 (window-level tracking) is tested in notebook 06, so this notebook corrects the two hypotheses it contains. The adjusted values are lower only because the family is smaller (floor 0.002 with 1,000 bootstrap draws). The conclusions do not change.
2. *Tie-breaking in the top-10.* The 0.1% to 0.4% drifts in the Kruskal H, the held-out cluster difference and mean nDCG (0.29581 to 0.29590 depending on the search routine) come from the 29 queries with an exact score tie at rank 10/11. The stable sort removes this dependence.
3. *Dependence on the grouping seed.* The cluster CIs use KMeans groups, so they could depend on its seed. Across five seeds the CI for rho(top-1, nDCG) moves only from 0.460 to 0.474 (lower end) and 0.567 to 0.575 (upper end); the design effect ranges 1.9 to 2.2. The conclusion does not hinge on the seed.

**Second-review changes (this revision).**
1. *R^2 point estimate and CI now describe the same quantity.* The earlier table paired the mean of five per-repeat R^2 values (0.275) with a bootstrap CI computed for the R^2 of the *averaged* prediction. The table now reports `r2_of_mean_pred` with its own CI, and keeps the mean of the repeats as a separate, labelled column. The two differ by at most 0.014 here (for example gbm fold-internal 0.311 vs 0.298), so the conclusion does not change, but the pair is now coherent.
2. *Fold-internal density features.* `nn_query_sim` and `q_density` are now recomputed inside each training fold for the CV rows. The transductive leak turned out to be immaterial: label-free ridge R^2 0.276 (transductive) vs 0.284 (fold-internal), gbm 0.317 vs 0.311; all CIs overlap almost completely.
3. *Coarser folds.* Holding out KMeans k = 10 groups gives ridge 0.273 (0.203 to 0.325) and gbm 0.298 (0.218 to 0.358): slightly lower and wider, as expected if neighbouring k = 50 clusters share topics across folds.
4. *Bootstrap p-values* now use `(1 + count)/(B + 1)`. *NaN bootstrap replicates* are counted: 0 were dropped anywhere in the notebook.
5. *Collinearity.* A drop-column check of the three score features together (held-out R^2 lost 0.276 ± 0.068 over folds) replaces reading the per-feature permutation importances as a ranking.
6. *"Why this test" notes* before each inferential section, and an in-sample signal-selection limitation (below).

## Findings
Numbers are from a full from-scratch run (seed 0, NFCorpus, multi-qa-MiniLM, 3,237 queries, mean nDCG@10 0.296; 33% of queries have nDCG 0).

- **Top-1 score is the strongest label-free correlate of success:** Spearman rho 0.52 (cluster CI 0.47 to 0.57); the top-1 gap is weaker (0.20, CI 0.16 to 0.24). Both H1 and H2 hold after Holm (adjusted p < 0.002, the resolution floor). H1 is close to mechanical, since top-1 and nDCG come from the same retrieval, so the size of rho is the informative part.
- **Query length and query density are weaker:** query tokens 0.06 (CI -0.004 to 0.11, BH-adjusted p 0.09, not significant), density 0.27.
- **The score signals are collinear** (VIF 8.6 for top-1, 11.6 for top-10 mean). Given the other two, top-1 keeps a partial rho of 0.33; the gap does not (-0.06). Controlling for the number of relevant documents and query length, top-1 keeps 0.48.
- **Label-free prediction is real but modest.** On unseen topic groups (k = 50 folds, density features computed inside the training fold), the R^2 of the averaged out-of-fold prediction is 0.284 (95% group-bootstrap CI 0.238 to 0.322) for ridge and 0.311 (0.266 to 0.354) for gradient boosting; permutation p < 0.01 (floor 0.0099); AUROC for failure about 0.80. With coarser k = 10 folds, R^2 is 0.273 (0.203 to 0.325) and 0.298 (0.218 to 0.358). Using all features including labelled ones gives R^2 0.47 to 0.59, which is close to the outcome by construction.
- **The transductive density features did not inflate R^2:** recomputing them inside training folds changed label-free R^2 by less than 0.01 (ridge 0.276 → 0.284, gbm 0.317 → 0.311).
- **Permutation importance** puts most weight on top-1 (R^2 drop 0.81), query length (0.09) and density (0.06), but under collinearity the split between score features is unreliable. Dropping all three score features together costs 0.28 ± 0.07 held-out R^2, so the score family as a whole carries most of the signal.
- **Circularity is small here:** the original and leave-one-out `rel_hubness` correlate 0.025 (CI -0.023 to 0.071) and -0.007 (CI -0.054 to 0.039) with nDCG.
- **The lowest cluster is lower on held-out queries too** (median difference -0.132, negative in 20 of 20 seeds), but cluster membership is only moderately stable (ARI 0.41 to 0.70, silhouette about 0.04 to 0.06).
- **CI width depends on the grouping:** for top-1, 0.052 (independent), 0.076 (k = 150), 0.102 (k = 50), 0.146 (k = 10). The shared-document grouping is unusable (one component holds 99.3% of queries).
- **No bootstrap replicate was dropped** as undefined (0 NaN replicates across the notebook).

### What this supports
On clean NFCorpus retrieval with this model, a query's top-1 similarity score is a moderate predictor of whether the query succeeds, no single other label-free signal adds much given it, and the out-of-sample estimate survives fold-internal features and coarser held-out groups.

### What this does not support
- That any feature *causes* success or failure (observational; confounded by the number of relevant documents and query type).
- That these correlations hold **under injected failures**. All features were measured on a clean system, so this is a screening result, not evidence that a detector will fire when retrieval degrades.
- **Unbiased downstream detector numbers.** Top-1 was chosen as *the* detector signal because of its correlation with nDCG on these same 3,237 queries, and notebook 06 then measures its power on the same queries. Tuning on NFCorpus is allowed, but the 06 power and AUROC numbers are in-sample for the choice of signal and are optimistic until the held-out run.
- That the ranking of signals carries to other datasets or models (earlier multi-dataset results showed density features changing sign). Those earlier results also mean the hypotheses here were not chosen blind to the held-out datasets (see the README).
- That the "hard cluster" is a stable, nameable topic, or that it generalizes to new topics: held-out queries share clusters and relevant documents with training queries.
- That the k = 50 CIs are the whole uncertainty: wider groupings give wider intervals, so treat them as lower bounds.
- That "label-free" features are fully production-like: the query set is restricted to queries with at least one relevant document.